## Multi-Layer Perceptron Example: Predicting Playoff Teams using Platoon Strength ##

Every NFL team's goal is to win the Super Bowl. To do so, you must first punch a ticket to the playoffs. This example seeks to predict playoff teams based on Offense, Defense, and Special Teams team strength. 

In [7]:
import numpy as np
import rice_ml
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

We will be using data from the 2015-2025 NFL seasons. We are interested in a binary classifer for playoff teams, and will be predicting it using our Offense, Defense, and Special Teams season-long Expected Points Added (EPA) per play. 

In [8]:
# Load in NFL receiving stats dataset
csv_path = Path("..") / "data" / "NFL_playoffs.csv"
data = pd.read_csv(csv_path)
# Display the first 5 rows of the dataset
data.head(5)

,Unnamed: 0,season,team,made_playoffs,home_wins,Off_epa_play,Def_epa_play,ST_epa_play,away_wins
0,1,2025,ARI,False,1.0,-0.022358,-0.081689,-0.108080,2.0
1,2,2025,ATL,False,4.0,-0.020990,-0.001011,-0.070223,4.0
2,3,2025,BAL,False,3.0,0.033055,-0.015021,-0.005860,5.0
3,4,2025,BUF,True,7.0,0.140667,0.009783,-0.068762,6.0
4,5,2025,CAR,True,5.0,-0.035640,-0.053701,0.014176,3.0


To clean our data, we first filter out columns we are not using (home_wins and away_wins). Then, we standardize the feature variables (Off, Def, ST epa/play). Finally, we encode our target vector (made_playoffs) as a binary classifier (1 if team made playoffs, 0 if not). 

We will be training our data set on teams before the 2025 season, and validating our results on the 2025 season. To do so, we split our data into a train and test set. 

Due to the implementation structure of the class, we need to one-hot encode the target variable and format them as lists of (n, 1) column vectors, where n is the number of possible outcomes. Likewise, the features need to be encoded as a list of (m, 1) column vectors, where m is the number of features. We do this after the train/test split. 

In [17]:
# Filter out home_wins and away_wins
filtered_data = data[["season", "team", "made_playoffs", "Off_epa_play", "Def_epa_play", "ST_epa_play"]]

# Drop any row that contains a NaN value for computational possibility
filtered_data = filtered_data.dropna()

# Standardize features
Off_std = rice_ml.StandardScaler()
filtered_data["Off_epa_play"] = Off_std.fit_transform(filtered_data[["Off_epa_play"]])
Def_std = rice_ml.StandardScaler()
filtered_data["Def_epa_play"] = Def_std.fit_transform(filtered_data[["Def_epa_play"]])
ST_std = rice_ml.StandardScaler()
filtered_data["ST_epa_play"] = ST_std.fit_transform(filtered_data[["ST_epa_play"]])

# Encode made_playoffs as binary
filtered_data["made_playoffs"] = filtered_data["made_playoffs"].astype(int)

# Display the first 5 rows of the dataset after encoding
filtered_data.head(5)

# Split data into train (pre 2025) and test (2025)
train_data = filtered_data[filtered_data["season"] < 2025]
print(f"Training samples available: {len(train_data)}")
test_data = filtered_data[filtered_data["season"] == 2025]
print(f"Training samples available: {len(test_data)}")

# Separate features and target variable
X = train_data[["Off_epa_play", "Def_epa_play", "ST_epa_play"]].values
y = train_data["made_playoffs"].values
X_test = test_data[["Off_epa_play", "Def_epa_play", "ST_epa_play"]].values
y_test = test_data["made_playoffs"].values

# One Hot Encode the target variable
encoder = rice_ml.OneHotEncoder()
y_train_matrix = encoder.fit_transform(y)
y_test_matrix = encoder.transform(y_test)
# Convert to lists of (2, 1) column vectors
y_train_encoded = [yi.reshape(-1, 1) for yi in y_train_matrix]
y_test_encoded = [yi.reshape(-1, 1) for yi in y_test_matrix]

# Convert features to lists of (3, 1) column vectors 
X_train_encoded = [xi.reshape(-1, 1) for xi in X]
X_test_encoded = [xi.reshape(-1, 1) for xi in X_test]

Training samples available: 312
Training samples available: 32


For our model, we will be using the MultiLayerPerceptron class in the rice_ml package. To intialize the model, we need to provide an arguement for our layers and neurons. For instance "[64, 64, 32]" indicates a 3 layer model with 64 neurons in the input layer, 64 in the second, and 32 in the outcome layer. For our model, we will use [3, 64, 32, 2]. This limits our inputs to 3 features, provides ample training opportunity, and limits our outcome to a binary classification.

To train our model, we simply call the train method. As arguements, we provide the feature array, target array, learning rate, and epochs. 

In [24]:
# Initialize MLP Model
mlp = rice_ml.MultiLayerPerceptron(layers = [3, 64, 32, 2])

# Train the model
mlp.train(X_train = X_train_encoded, y_train = y_train_encoded, epochs = 5, alpha = 0.046)



Starting Cost = 0.2710770371237278
1-Epoch Cost = 0.21308914696503323
2-Epoch Cost = 0.18048284328982278
3-Epoch Cost = 0.15172671252117864
4-Epoch Cost = 0.1335001742518816
5-Epoch Cost = 0.12344672331809889


As we can see from the printed cost results, each epoch reduces the final cost. The reduction starts to flatten around 5 epochs, so we can stop there.

Now we are ready to test the model. To do so, we will use the predict method on our 2025 season feature data. Then, we can compare predicted results to the actual 2025-26 season playoffs. Note that the predict method is designed to predict one entry at a time. We will use list comprehension to predict all the points in the test set.

In [29]:
# Loop through the test set and predict one by one
predictions = [mlp.predict(xi) for xi in X_test_encoded]
# Add predictions to the test dataframe
results_df = test_data.copy()
results_df["predicted_playoffs"] = predictions

# View the Team, Actual Result, and Predicted Result
comparison = results_df[["team", "made_playoffs", "predicted_playoffs"]]
print(comparison)

# Compute accuracy, precision, recall
accuracy = rice_ml.accuracy_score(results_df["made_playoffs"], results_df["predicted_playoffs"])
precision = rice_ml.precision_score(results_df["made_playoffs"], results_df["predicted_playoffs"])
recall = rice_ml.recall_score(results_df["made_playoffs"], results_df["predicted_playoffs"])
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")  


   team  made_playoffs  predicted_playoffs
0   ARI              0                   0
1   ATL              0                   0
2   BAL              0                   0
3   BUF              1                   1
4   CAR              1                   0
5   CHI              1                   1
6   CIN              0                   0
7   CLE              0                   0
8   DAL              0                   0
9   DEN              1                   1
10  DET              0                   1
11   GB              1                   1
12  HOU              1                   1
13  IND              0                   1
14  JAX              1                   1
15   KC              0                   0
16   LA              1                   1
17  LAC              1                   0
18   LV              0                   0
19  MIA              0                   0
20  MIN              0                   0
21   NE              1                   1
22   NO    

At a glance, all three of Accuracy, Precision, and Recall are relatively high, indicating a very strong model. It is important to analyze the errors to identify model limitations or hidden strengths.

The teams that made the playoffs though they were not predicted to are PIT, LAC, and CAR. These are Type II errors. All of these teams were very weak playoff teams. The Chargers (LAC) were the last (7th) seed in the AFC and won several games late in the season due to lucky bounces. Both Carolina (CAR) and Pittsburgh (PIT) were the weakest division winners in their conference, and played in very poor divisions. These teams would not have qualified for the playoffs had they not won their division. Overall, none of these teams were "playoff-caliber" in a team strength sense. It is fair to say that the model is sound in predicting who will not make, or should not be in, the playoffs.

The Type I errors are teams who the model says should be in the playoffs but are not. These teams are Detroit (DET) and Indianapolis (IND). Detroit was a good team in a strong division. They missed the playoffs by 0.5 wins (the Packers had tied a game) and had several near wins throughout the season. Had they gotten lucky and won any of those games, they would have made the playoffs. The Indianapolis Colts were the best team in the AFC for close to half of the season before their starting quarterback got injured. They did not win any games in the second half of the year. This error highlights a key limitation of the model: the team strength metric does not account for changes in strength throughout the season. Due to this limitation, the first half of the Colts' season is skewing their strength and making them appear to be a very strong team. However, it important to note that while the Colts did miss the playoffs, they were still a wildcard possibility late into the year. Thus, despite this limitation, the model is not completely wrong in these edge cases. 

Overall, this model is very strong, and can effectively predict whether an NFL team will, or should be in, the playoffs. 